# VERITAS Thesis Implementation - Environment Setup

This notebook sets up the foundational environment for the VERITAS (Vision-based EfficientNet-Reinforced Image Tampering Authentication and ASPP-Segmentation MTL-Framework) thesis implementation.

**Task 1: Setup project structure and Google Colab environment**
- Create directory structure on Google Drive
- Mount Google Drive in Colab notebook
- Install required dependencies
- Verify CUDA availability and log GPU model/VRAM
- Create configuration management system
- Set up reproducibility (fixed random seeds)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define base path for VERITAS project
BASE_PATH = '/content/drive/MyDrive/VERITAS/'

print(f"✓ Google Drive mounted successfully")
print(f"✓ Base path: {BASE_PATH}")

## 2. Create Directory Structure

In [ ]:
import os

# Define directory structure
directories = {
    'dataset': 'Dataset storage (images and annotations)',
    'checkpoints': 'Model checkpoint storage',
    'logs': 'Training logs and metrics',
    'results': 'Evaluation results and visualizations'
}

# Create directories
for dir_name, description in directories.items():
    dir_path = os.path.join(BASE_PATH, dir_name)
    os.makedirs(dir_path, exist_ok=True)
    print(f"✓ Created: {dir_path}")
    print(f"  Purpose: {description}")

print("\n✓ Directory structure created successfully")

## 3. Install Required Dependencies

In [ ]:
# Install required packages
# Note: PyTorch and torchvision are pre-installed in Colab, but we'll verify versions

import sys

print("Installing required dependencies...\n")

# Install packages that might not be pre-installed or need specific versions
!pip install -q opencv-python-headless
!pip install -q scikit-learn
!pip install -q scipy
!pip install -q Pillow

print("\n✓ Dependencies installed successfully")

## 4. Verify Installation and Log Environment

In [ ]:
import torch
import torchvision
import cv2
import numpy as np
import sklearn
import scipy
from PIL import Image
import sys

# Print Python version
print("="*60)
print("ENVIRONMENT VERIFICATION")
print("="*60)

print(f"\nPython version: {sys.version.split()[0]}")

# Print package versions
print(f"\nDependency Versions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  torchvision: {torchvision.__version__}")
print(f"  OpenCV: {cv2.__version__}")
print(f"  NumPy: {np.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  scipy: {scipy.__version__}")
print(f"  Pillow: {Image.__version__}")

print("\n✓ All required packages installed and verified")

## 5. CUDA and GPU Verification

In [ ]:
import torch
import subprocess

print("="*60)
print("GPU AND CUDA VERIFICATION")
print("="*60)

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"\nCUDA Available: {cuda_available}")

if cuda_available:
    # Get GPU information
    gpu_count = torch.cuda.device_count()
    print(f"GPU Count: {gpu_count}")
    
    for i in range(gpu_count):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)  # Convert to GB
        
        print(f"\nGPU {i}:")
        print(f"  Model: {gpu_name}")
        print(f"  Total VRAM: {gpu_memory:.2f} GB")
    
    # Get CUDA version
    cuda_version = torch.version.cuda
    print(f"\nCUDA Version: {cuda_version}")
    
    # Get cuDNN version
    cudnn_version = torch.backends.cudnn.version()
    print(f"cuDNN Version: {cudnn_version}")
    
    # Test GPU with a simple operation
    try:
        test_tensor = torch.randn(100, 100).cuda()
        result = torch.matmul(test_tensor, test_tensor)
        print(f"\n✓ GPU test successful - device is functioning correctly")
    except Exception as e:
        print(f"\n✗ GPU test failed: {e}")
else:
    print("\n⚠ WARNING: CUDA is not available. Running on CPU.")
    print("  For GPU acceleration, ensure you're using a GPU runtime:")
    print("  Runtime > Change runtime type > Hardware accelerator > GPU")

# Store GPU info for later use
gpu_info = {
    'cuda_available': cuda_available,
    'gpu_count': torch.cuda.device_count() if cuda_available else 0,
    'gpu_name': torch.cuda.get_device_name(0) if cuda_available else 'CPU',
    'total_vram_gb': torch.cuda.get_device_properties(0).total_memory / (1024**3) if cuda_available else 0,
    'cuda_version': torch.version.cuda if cuda_available else 'N/A'
}

print("\n" + "="*60)

## 6. Configuration Management System

In [ ]:
import json
import os
from datetime import datetime

# Create default configuration
default_config = {
    "project_name": "VERITAS",
    "version": "1.0.0",
    "created_at": datetime.now().isoformat(),
    
    # Paths
    "paths": {
        "base_path": BASE_PATH,
        "dataset_path": os.path.join(BASE_PATH, "dataset"),
        "checkpoints_path": os.path.join(BASE_PATH, "checkpoints"),
        "logs_path": os.path.join(BASE_PATH, "logs"),
        "results_path": os.path.join(BASE_PATH, "results")
    },
    
    # Model configuration
    "model": {
        "input_resolution": 600,
        "backbone": "efficientnet_b7",
        "pretrained": True,
        "num_classes": 1
    },
    
    # Training configuration
    "training": {
        "batch_size": 8,
        "num_epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "optimizer": "adamw",
        "gradient_clip_max_norm": 1.0
    },
    
    # Loss weights
    "loss": {
        "classification_weight": 0.4,
        "segmentation_weight": 0.6,
        "segmentation_bce_weight": 0.5,
        "segmentation_dice_weight": 0.5
    },
    
    # Data split
    "data": {
        "train_split": 0.7,
        "val_split": 0.15,
        "test_split": 0.15,
        "supported_categories": [
            "authentic",
            "face_swap",
            "face_reenact",
            "inpainting"
        ]
    },
    
    # Augmentation configuration
    "augmentation": {
        "enabled": True,
        "rotation_degrees": 2.0,
        "width_shift_range": 0.1,
        "height_shift_range": 0.1,
        "shear_range": 0.1,
        "zoom_range": 0.05,
        "horizontal_flip": True
    },
    
    # Representation configuration
    "representations": {
        "rgb_normalization": "imagenet",
        "imagenet_mean": [0.485, 0.456, 0.406],
        "imagenet_std": [0.229, 0.224, 0.225],
        "srm_filter_count": 3,
        "noise_normalization": "tanh",
        "dct_block_size": 8,
        "frequency_bands": ["low", "mid", "high"]
    },
    
    # ASPP configuration
    "aspp": {
        "dilation_rates": [1, 6, 12, 18],
        "output_channels": 256
    },
    
    # Reproducibility
    "random_seed": 42,
    
    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers": 4,
    
    # Environment information
    "environment": gpu_info
}

# Save configuration to Google Drive
config_path = os.path.join(BASE_PATH, 'config.json')
with open(config_path, 'w') as f:
    json.dump(default_config, f, indent=2)

print("="*60)
print("CONFIGURATION MANAGEMENT")
print("="*60)
print(f"\n✓ Configuration file created: {config_path}")
print(f"\nConfiguration summary:")
print(f"  Model: {default_config['model']['backbone']}")
print(f"  Input resolution: {default_config['model']['input_resolution']}x{default_config['model']['input_resolution']}")
print(f"  Batch size: {default_config['training']['batch_size']}")
print(f"  Learning rate: {default_config['training']['learning_rate']}")
print(f"  Device: {default_config['device']}")
print(f"  Random seed: {default_config['random_seed']}")

## 7. Setup Reproducibility

In [ ]:
import random
import numpy as np
import torch

def set_random_seed(seed=42):
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    
    Args:
        seed (int): Random seed value
    """
    # Python random
    random.seed(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU
    
    # Set deterministic behavior (may impact performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✓ Random seed set to {seed} for reproducibility")
    print(f"  - Python random: {seed}")
    print(f"  - NumPy: {seed}")
    print(f"  - PyTorch: {seed}")
    print(f"  - cuDNN deterministic: True")
    print(f"  - cuDNN benchmark: False")

# Set the random seed from configuration
set_random_seed(default_config['random_seed'])

## 8. Configuration Loading Utility

In [ ]:
def load_config(config_path):
    """
    Load configuration from JSON file.
    
    Args:
        config_path (str): Path to configuration JSON file
    
    Returns:
        dict: Configuration dictionary
    """
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    return config

def save_config(config, config_path):
    """
    Save configuration to JSON file.
    
    Args:
        config (dict): Configuration dictionary
        config_path (str): Path to save configuration JSON file
    """
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"✓ Configuration saved to {config_path}")

def update_config(config_path, updates):
    """
    Update existing configuration with new values.
    
    Args:
        config_path (str): Path to configuration JSON file
        updates (dict): Dictionary of updates to apply
    
    Returns:
        dict: Updated configuration
    """
    config = load_config(config_path)
    
    # Deep update
    def deep_update(base_dict, update_dict):
        for key, value in update_dict.items():
            if isinstance(value, dict) and key in base_dict and isinstance(base_dict[key], dict):
                deep_update(base_dict[key], value)
            else:
                base_dict[key] = value
    
    deep_update(config, updates)
    save_config(config, config_path)
    
    return config

print("✓ Configuration management utilities defined")
print("  - load_config(config_path)")
print("  - save_config(config, config_path)")
print("  - update_config(config_path, updates)")

## 9. Environment Summary

In [ ]:
print("="*60)
print("VERITAS ENVIRONMENT SETUP COMPLETE")
print("="*60)

print("\n✓ Google Drive mounted")
print("✓ Directory structure created")
print("✓ Dependencies installed and verified")
print("✓ GPU/CUDA verified")
print("✓ Configuration management system created")
print("✓ Reproducibility configured (random seed: 42)")

print("\n" + "="*60)
print("DIRECTORY STRUCTURE")
print("="*60)

for dir_name in directories.keys():
    dir_path = os.path.join(BASE_PATH, dir_name)
    print(f"  {dir_path}")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)

print("\n1. Upload OpenForensics dataset to Google Drive")
print(f"   Location: {os.path.join(BASE_PATH, 'dataset')}")

print("\n2. Proceed to Task 2: Implement dataset ingestion and validation pipeline")

print("\n3. Configuration can be loaded at any time with:")
print(f"   config = load_config('{config_path}')")

print("\n" + "="*60)